# Experiment 2 — three corrections

Recomputations on the **saved** artifacts of the first ablation run. Nothing is retrained; the dual is
re-solved on the same pool and reproduces the same weights. Pre-registered in `taskc/DECISIONS.md`
section 24.4, written before this ran.

1. **SIR timing** — it drew a 10⁶ pool to resample 10⁵, so its 97.8 s mostly measures drawing 10× more
   paths. Re-timed at a **matched 10⁵ pool**, with draw / reweight / resample reported separately.
2. **Exotics against Q\***, not Heston-Q. SMT, (b) and SIR target the projection of P_θ onto the
   constraints; scoring them against Heston-Q charges them for the Q\*-to-Heston-Q gap, which is a
   property of the projection rather than of the sampler.
3. **Columns against Q\***, for the same reason, for all four arms.

Arm (a) was trained on Heston-Q paths, so **Heston-Q is its correct target** and Q\* is not — it is
flagged wherever it appears in the Q\* tables.

In [ ]:
PINNED_COMMIT    = "d4407599522ac7eace38fc8b3fbab52921052bf0"
NOTEBOOK_VERSION = "corr-2026.09.25b"
EXPECT_TASKC     = "taskc-2026.09.25b"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment and the existing artifacts

In [ ]:
import os, sys, json, math, time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import taskc
from taskc.config import frozen
from taskc.ptheta import load_checkpoint, make_schedule, sample_ptheta
from taskc.smt import resample_weighted, sliced_wasserstein
from taskc.dual import solve_level
from taskc.gate import all_columns
from config import q_params, CALIB_TESTFUNS, VANILLA_C3, EXOTICS
from constraints import build
import evaluation as ev

_nb_path = "notebooks/taskc_11_ablation_corrections_colab.ipynb"
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb_path).read(), (
    f"{NOTEBOOK_VERSION} does not appear in the copy of this notebook at PINNED_COMMIT "
    f"({PINNED_COMMIT[:7]}). Stale cached notebook, or the pin was not advanced with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_ablation"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/ablation_local")
print("taskc:", taskc.__version__, "| device:", DEVICE, "| tf32:", torch.backends.cuda.matmul.allow_tf32)

for f in ("ablation.json", "pool_z.npz", "arm_draws.npz", "frozen/ptheta_mlp21.pt"):
    p = os.path.join(DRIVE, f)
    assert os.path.exists(p), f"missing {p} -- run taskc_09_ablation_colab.ipynb first"
prev = json.load(open(os.path.join(DRIVE, "ablation.json")))
N_POOL, N_EVAL = prev["n_pool"], prev["n_eval"]
SKIP_DONE = True
RUN = frozen(artifact_dir=DRIVE)
RES = os.path.join(DRIVE, "ablation_corrections.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__,
    device=DEVICE, of_run=prev.get("notebook"), stages={}, timings={})
def save(): json.dump(res, open(RES, "w"), indent=1, default=float)
save()

model, std, ck_cfg, _ = load_checkpoint(os.path.join(DRIVE, "frozen/ptheta_mlp21.pt"), device=DEVICE)
sched = make_schedule(RUN, device=DEVICE)
q = q_params()
z_pool = np.load(os.path.join(DRIVE, "pool_z.npz"))["z"].astype(np.float64)
pool_paths = std.to_paths(z_pool, world="Ptheta_pool")
tilt, rsolve, cs_pool = solve_level("C3", pool_paths, q)
w = rsolve.w
print(f"pool {z_pool.shape}  dual reproduced: ESS {rsolve.ess_frac*100:.2f}%  KL {rsolve.kl:.4f}")
D = np.load(os.path.join(DRIVE, "arm_draws.npz"))
ARMS = {"SMT": D["smt"], "a_qtrained": D["a"], "b_lstar": D["b"], "SIR": D["sir"]}
print({k: v.shape for k, v in ARMS.items()})

### Correction 1 — SIR at a matched 10⁵ pool

Timing split three ways. Note the tension recorded in DECISIONS.md 24.4: at a matched pool SIR's
statistical quality **must** get worse, because 10⁵ draws resampled from a 10⁵ pool duplicate heavily.
Timing and quality cannot both be held fixed, so both pools are reported and labelled.

In [ ]:
if "sir_matched" not in res["stages"]:
    t0 = time.time()
    t = time.time()
    d_m = sample_ptheta(model, sched, n=N_EVAL, seed=6001, cfg=RUN, device=DEVICE, verbose=False)
    t_draw = time.time()-t
    t = time.time()
    p_m = std.to_paths(d_m.z.astype(np.float64), world="Ptheta_matched")
    cs_m = build(p_m, CALIB_TESTFUNS, VANILLA_C3, q)
    w_m = tilt.weights(cs_m.G)
    t_reweight = time.time()-t
    t = time.time()
    z_m = resample_weighted(d_m.z.astype(np.float64), w_m, N_EVAL, seed=6002)
    t_resample = time.time()-t
    res["stages"]["sir_matched"] = dict(
        n_pool=N_EVAL, draw_s=t_draw, reweight_s=t_reweight, resample_s=t_resample,
        total_s=t_draw+t_reweight+t_resample,
        ess=float(1/np.sum(w_m**2)/len(w_m)),
        unique_frac=float(len(np.unique(z_m, axis=0))/N_EVAL))
    np.savez_compressed(os.path.join(DRIVE, "sir_matched.npz"), z=z_m.astype(np.float32))
    res["timings"]["sir_matched"] = time.time()-t0; save()
S = res["stages"]["sir_matched"]
old = prev["stages"]["draws"]["sample_seconds"]["SIR"]
print(f"SIR at a MATCHED {N_EVAL:,} pool: draw {S['draw_s']:.1f}s + reweight {S['reweight_s']:.1f}s "
      f"+ resample {S['resample_s']:.2f}s = {S['total_s']:.1f}s")
print(f"   ESS {S['ess']*100:.2f}%   unique draws {S['unique_frac']*100:.1f}%")
print(f"original {N_POOL:,}-pool figure was {old:.1f}s with unique "
      f"{prev['stages']['draws']['sir_unique']*100:.1f}%  <- NOT matched; it drew 10x more paths")
ARMS["SIR_matched"] = np.load(os.path.join(DRIVE, "sir_matched.npz"))["z"]

### Corrections 2 and 3 — everything against the weighted Q\* on the same pool

The reference for every quantity is the self-normalised weighted estimate on the 10⁶ pool, and its own
SNIS error is carried in quadrature.

In [ ]:
G_pool, names, kinds, calibrated = all_columns(pool_paths)
ref_col = w @ G_pool
ref_col_se = np.sqrt(np.sum(w[:,None]**2 * (G_pool - ref_col[None,:])**2, axis=0))
ref_ex = {}
for k, sp in EXOTICS.items():
    pay = ev.exotic_payoff(pool_paths, **sp)
    m = float(w @ pay)
    ref_ex[k] = dict(price=m, se=float(np.sqrt(np.sum(w**2 * (pay - m)**2))))
print("weighted Q* reference on the pool:")
for k, v in ref_ex.items(): print(f"  {k:28s} {v['price']:9.5f} +- {v['se']:.5f}")

if "vs_qstar" not in res["stages"]:
    t0 = time.time(); out = {}
    for arm, z in ARMS.items():
        p = std.to_paths(z.astype(np.float64), world=arm)
        G, _, _, _ = all_columns(p)
        n = G.shape[0]
        mean = G.mean(0); se = G.std(0)/math.sqrt(n)
        comb = np.sqrt(se**2 + ref_col_se**2)
        zc = (mean - ref_col)/np.where(comb > 0, comb, np.inf)
        ex = {}
        for k, sp in EXOTICS.items():
            pay = ev.exotic_payoff(p, **sp)
            m = float(pay.mean()); s = float(pay.std(ddof=1)/math.sqrt(len(pay)))
            r = ref_ex[k]
            ex[k] = dict(price=m, se=s, diff=m-r["price"],
                         diff_se=(m-r["price"])/math.sqrt(s**2 + r["se"]**2),
                         diff_pct=100*(m-r["price"])/r["price"])
        out[arm] = dict(
            calib=dict(max_abs_z=float(np.abs(zc[calibrated]).max()),
                       rms_z=float(np.sqrt((zc[calibrated]**2).mean())),
                       worst=str(np.array(names)[calibrated][int(np.argmax(np.abs(zc[calibrated])))])),
            heldout=dict(max_abs_z=float(np.abs(zc[~calibrated]).max()),
                         rms_z=float(np.sqrt((zc[~calibrated]**2).mean())),
                         worst=str(np.array(names)[~calibrated][int(np.argmax(np.abs(zc[~calibrated])))])),
            exotics=ex)
    res["stages"]["vs_qstar"] = dict(arms=out, ref_exotics=ref_ex)
    res["timings"]["vs_qstar"] = time.time()-t0; save()
print("recomputed")

### The corrected tables

In [ ]:
V = res["stages"]["vs_qstar"]["arms"]; M_old = prev["stages"]["metrics"]
LAB = {"SMT":"SMT","a_qtrained":"(a) Q-trained [TARGET SAMPLES; its target is Heston-Q, NOT Q*]",
       "b_lstar":"(b) L*-resampled","SIR":"SIR (1e6 pool)","SIR_matched":"SIR (matched 1e5 pool)"}
print("="*112)
print("COLUMNS vs weighted Q* on the same pool   (was: vs exact Heston-Q)")
print("="*112)
print(f"{'arm':52s} {'calib max/rms':>18s} {'held-out max/rms':>20s}   {'calib rms vs Heston-Q':>21s}")
for a in ARMS:
    v = V[a]; old = M_old.get(a, {}).get("calib", {}).get("rms_z")
    print(f"{LAB[a]:52s} {v['calib']['max_abs_z']:8.2f}/{v['calib']['rms_z']:7.2f} "
          f"{v['heldout']['max_abs_z']:10.2f}/{v['heldout']['rms_z']:7.2f}   "
          f"{('%21.2f'%old) if old is not None else '':>21s}")
print(f"\nworst calibrated column: " + ", ".join(f"{a}={V[a]['calib']['worst']}" for a in ARMS))

print("\n"+"="*112)
print("EXOTICS vs weighted Q* on the same pool, in SE units   (was: vs Heston-Q)")
print("="*112)
R = res["stages"]["vs_qstar"]["ref_exotics"]
print(f"{'exotic':28s} " + "".join(f"{LAB[a].split(' ')[0]:>20s}" for a in ARMS) + f" {'Q* ref':>11s}")
for k in EXOTICS:
    print(f"{k:28s} " + "".join(f"{V[a]['exotics'][k]['price']:11.5f}({V[a]['exotics'][k]['diff_se']:+6.2f})"
                                for a in ARMS) + f" {R[k]['price']:11.5f}")
print("\nvalues are price(difference from weighted Q* in SE). Arm (a) targets Heston-Q, not Q*.")
# Per-arm summary. The bound is attained by ONE exotic, so name it and count how many
# exotics fall inside each band, instead of describing the max as if it applied to all.
print()
for a in ARMS:
    ses = {k: V[a]["exotics"][k]["diff_se"] for k in EXOTICS}
    worst = max(ses, key=lambda k: abs(ses[k]))
    n1 = sum(abs(v) <= 1 for v in ses.values())
    n2 = sum(abs(v) <= 2 for v in ses.values())
    detail = ", ".join(f"{k[:12]} {v:+.2f}" for k, v in ses.items())
    print(f"  {a:12s} largest |diff| {abs(ses[worst]):5.2f} SE on {worst}; "
          f"{n1}/{len(ses)} within 1 SE, {n2}/{len(ses)} within 2 SE   [{detail}]")
print("\ntimings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `ablation_corrections.json`. Outcomes go into DECISIONS.md section 24.R against the
section 24.3 predictions, including any that are falsified.

### Correction 4 — is SMT's deviation bias or a single-draw fluctuation?

Pre-registered in `taskc/DECISIONS.md` section 24.5 before running. Five independent 10^5 draws with
fresh seeds from SMT, (b) and SIR at a matched 10^5 pool. Nothing is retrained; the saved pool and dual
are reused. SIR gets a **fresh pool per draw**, or the five would share one resampling error.

**Decision rule, fixed in advance:** under the null the per-draw difference has unit SE, so the mean of
five has an across-draw SE of about 0.45. A mean further than **3 across-draw SEs** from zero is bias;
within 2 is noise; between 2 and 3 is inconclusive and reported as such.

In [ ]:
N_REP = 5
if "repeat" not in res["stages"]:
    t0 = time.time()
    from taskc.hnet import load_hnet
    from taskc.smt import sample_smt
    hnet = load_hnet(os.path.join(DRIVE, "frozen/hpsi_C3.pt"), device=DEVICE)
    m_b, _, _, _ = load_checkpoint(os.path.join(DRIVE, "ptheta_lstar.pt"), device=DEVICE)
    rep = {a: {k: [] for k in EXOTICS} for a in ("SMT", "b_lstar", "SIR_matched")}
    for i in range(N_REP):
        dr = {}
        dr["SMT"] = sample_smt(model, hnet, sched, N_EVAL, 7100 + i, RUN, DEVICE, verbose=False).z
        dr["b_lstar"] = sample_ptheta(m_b, sched, n=N_EVAL, seed=7200 + i, cfg=RUN,
                                      device=DEVICE, verbose=False).z
        dp = sample_ptheta(model, sched, n=N_EVAL, seed=7300 + i, cfg=RUN, device=DEVICE, verbose=False)
        csi = build(std.to_paths(dp.z.astype(np.float64)), CALIB_TESTFUNS, VANILLA_C3, q)
        dr["SIR_matched"] = resample_weighted(dp.z.astype(np.float64), tilt.weights(csi.G),
                                              N_EVAL, seed=7400 + i)
        for a, z in dr.items():
            pth = std.to_paths(np.asarray(z, dtype=np.float64), world=a)
            for k, sp in EXOTICS.items():
                pay = ev.exotic_payoff(pth, **sp)
                m = float(pay.mean()); s = float(pay.std(ddof=1) / math.sqrt(len(pay)))
                r = ref_ex[k]
                rep[a][k].append((m - r["price"]) / math.sqrt(s ** 2 + r["se"] ** 2))
        print(f"  draw {i+1}/{N_REP} done ({time.time()-t0:.0f}s)", flush=True)
    res["stages"]["repeat"] = dict(n_rep=N_REP, per_draw_se=rep)
    res["timings"]["repeat"] = time.time() - t0
    save()

RP = res["stages"]["repeat"]["per_draw_se"]
print("\n" + "=" * 106)
print(f"{res['stages']['repeat']['n_rep']} independent 1e5 draws -- difference from weighted Q*, in SE")
print("=" * 106)
print(f"{'arm':14s} {'exotic':28s} {'mean':>7s} {'SE(mean)':>9s} {'mean/SE':>8s}  {'verdict':>13s}  per-draw")
for a in ("SMT", "b_lstar", "SIR_matched"):
    for k in EXOTICS:
        v = np.asarray(RP[a][k], dtype=float)
        mu = float(v.mean()); sem = float(v.std(ddof=1) / math.sqrt(len(v)))
        ratio = mu / sem if sem > 0 else float("inf")
        verdict = "BIAS" if abs(ratio) > 3 else "noise" if abs(ratio) < 2 else "inconclusive"
        per = ", ".join(f"{x:+.2f}" for x in v)
        print(f"{a:14s} {k:28s} {mu:+7.2f} {sem:9.2f} {ratio:+8.2f}  {verdict:>13s}  [{per}]")
print("\nrule (pre-registered 24.5): |mean| > 3 across-draw SE = bias; < 2 = noise; between = inconclusive")